# Week 17 · Notebook 1 — A Small Eval Harness (LLM-as-Judge)
**CSCI 250 — Introduction to Artificial Intelligence · Fall 2026**

Build a tiny **eval harness**: test cases + scorers + an aggregate report. We score with **exact-match** *and* an **LLM-as-judge** (Claude, with a Gemini alternative). **No OpenAI.**

> Degrades gracefully: without an API key the judge falls back to a clear message, and exact-match still runs offline.

## 0. Install + load keys safely

In [ ]:
!pip -q install anthropic google-generativeai
import os
try:
    from google.colab import userdata
    os.environ['ANTHROPIC_API_KEY'] = userdata.get('ANTHROPIC_API_KEY')
    os.environ['GEMINI_API_KEY'] = userdata.get('GEMINI_API_KEY')
except Exception:
    pass  # locally, set these in your shell environment
print('Anthropic key set:', bool(os.environ.get('ANTHROPIC_API_KEY')))
print('Gemini key set:', bool(os.environ.get('GEMINI_API_KEY')))

## 1. The system under test
Any function `input -> output` works. We use Claude, but you would plug in your Final Project pipeline (RAG, agent, fine-tuned model, ...).

In [ ]:
MODEL = 'claude-haiku-4-5-20251001'

def system_under_test(question: str) -> str:
    try:
        import anthropic
        client = anthropic.Anthropic()
        msg = client.messages.create(
            model=MODEL, max_tokens=100,
            messages=[{'role': 'user',
                       'content': f'Answer briefly: {question}'}])
        return msg.content[0].text.strip()
    except Exception as e:
        return f'[offline/no key: {e}]'

print(system_under_test('What is the capital of France?'))

## 2. Test cases
Each case has an input and a reference of what 'good' looks like. 5-10 cases is enough to start; grow them as you find failures.

In [ ]:
test_cases = [
    {'input': 'What is the capital of France?', 'expected': 'Paris'},
    {'input': 'What is 2 + 2?',                 'expected': '4'},
    {'input': 'What language is this course taught in?', 'expected': 'Python'},
    {'input': 'Who wrote Romeo and Juliet?',    'expected': 'Shakespeare'},
]
print(len(test_cases), 'test cases')

## 3. Scorer A — exact-match / contains
Objective and free. Good for single-answer tasks; brittle for free-form text ('Paris.' vs 'The capital is Paris').

In [ ]:
def contains_score(answer: str, expected: str) -> float:
    return 1.0 if expected.lower() in answer.lower() else 0.0

print(contains_score('The capital is Paris.', 'Paris'))  # 1.0
print(contains_score('I am not sure.', 'Paris'))         # 0.0

## 4. Scorer B — LLM-as-judge (Claude)
A strong model grades against a rubric. We ask for a single number and keep temperature low. **Judge hygiene:** clear rubric, structured output, spot-check against your own judgment, and remember judges can favor longer/own-style answers.

In [ ]:
def judge_score(answer: str, expected: str, question: str = '') -> float:
    prompt = (
        'You are a strict grader. Score the model answer for correctness.\n'
        f'Reference answer: {expected}\n'
        f'Model answer: {answer}\n'
        'Reply with ONLY 1 (correct) or 0 (incorrect).')
    try:
        import anthropic
        client = anthropic.Anthropic()
        out = client.messages.create(
            model='claude-sonnet-4-6', max_tokens=5, temperature=0,
            messages=[{'role': 'user', 'content': prompt}]).content[0].text
        return 1.0 if '1' in out else 0.0
    except Exception as e:
        print('   (judge offline:', e, ')')
        return float('nan')

print('judge demo:', judge_score('It is Paris.', 'Paris'))

### Gemini judge alternative
Same idea with Gemini — useful for a second opinion or if you only have a Gemini key.

In [ ]:
def gemini_judge(answer: str, expected: str) -> float:
    try:
        import google.generativeai as genai
        genai.configure(api_key=os.environ['GEMINI_API_KEY'])
        model = genai.GenerativeModel('gemini-2.5-flash')
        prompt = (f'Reference: {expected}\nAnswer: {answer}\n'
                  'Reply ONLY 1 if the answer is correct, else 0.')
        out = model.generate_content(prompt).text
        return 1.0 if '1' in out else 0.0
    except Exception as e:
        print('   (gemini judge offline:', e, ')')
        return float('nan')

## 5. The harness — run, score, aggregate
This is the whole point: one function that runs every case, scores it, and returns an average plus a per-case table.

In [ ]:
def run_eval(system_fn, score_fn):
    rows, scores = [], []
    for tc in test_cases:
        answer = system_fn(tc['input'])
        s = score_fn(answer, tc['expected'])
        rows.append((tc['input'], answer, tc['expected'], s))
        if s == s:  # skip NaN (judge offline)
            scores.append(s)
    avg = sum(scores) / len(scores) if scores else float('nan')
    return avg, rows

avg, rows = run_eval(system_under_test, contains_score)
print(f'\nExact-match average: {avg:.2f}\n')
for q, a, exp, s in rows:
    print(f'  [{s}] Q: {q}\n       A: {a}\n       expected ~ {exp}\n')

## 6. Run the LLM-as-judge eval
Re-run with the judge scorer. Compare its average to exact-match — the judge often credits correct-but-differently-worded answers.

In [ ]:
judge_avg, judge_rows = run_eval(system_under_test,
                                 lambda a, e: judge_score(a, e))
print(f'LLM-judge average: {judge_avg:.2f}')

## 7. Track it over time
Save the score with a label (model + prompt version). Re-run after every change to prove an improvement — and to justify a cheaper model.

In [ ]:
import datetime
record = {'when': datetime.datetime.now().isoformat(timespec='minutes'),
          'model': MODEL, 'exact_match': round(avg, 3),
          'judge': None if judge_avg != judge_avg else round(judge_avg, 3)}
print('eval record:', record)
# Append records to a CSV/JSON in your repo to chart quality over versions.

## 8. Use this in your Final Project
- Swap `system_under_test` for **your** pipeline.
- Add 5-10 test cases that matter for your problem.
- Report the average score in your write-up.
- Pair this with the **safety checklist** (hallucination, bias, prompt injection, privacy) from this week's document.